# Lecture 8.7 — Caching Expensive Calls with `before_tool_callback` + `after_tool_callback`

**Design Pattern:** Caching (P4)
**Callbacks used:** `before_tool_callback` + `after_tool_callback` (used as a pair)
**Applied to:** `cost_cutter_agent` (on the `google_search_tool`)

In this lecture we implement a **full read-through cache** for `google_search_tool` using session state as the cache store. The two callbacks work as a pair:

- `before_tool_callback` → **cache read**: checks `tool_context.state` for a cached result; on a hit, returns it immediately and skips the tool entirely
- `after_tool_callback` → **cache write**: on a miss (the tool just executed), stores the fresh result into state under the same key

The three-part execution flow:
1. **Cache key generation** — a deterministic string built from tool name + serialised arguments (`json.dumps(args, json.dumps(args))`)
2. **Cache read** (`read_search_cache`) — on hit, returns the cached dict and skips the tool; on miss, returns `None` and lets the tool run
3. **Cache write** (`write_search_cache`) — stores the fresh result under the key after a real tool execution

---
**Changes from Lecture 8.6 (the complete diff):**
1. `import json` added to the imports cell (needed for `json.dumps` in key generation)
2. New shared helper: `_make_cache_key(tool_name, args)` — deterministic key construction
3. New function: `read_search_cache(tool, args, tool_context)` — the `before_tool_callback` (cache read)
4. New function: `write_search_cache(tool, args, tool_context, tool_response)` — the `after_tool_callback` (cache write)
5. `cost_cutter_agent` gains two new keyword arguments: `before_tool_callback=read_search_cache` and `after_tool_callback=write_search_cache`
6. New standalone demo cell — all four cache scenarios with mock objects
7. New timed full-workflow cell — `time.time()` before and after, showing concrete speed savings across loop iterations

---
**Critical insight — the asymmetry:** On a **cache hit**, `after_tool_callback` (the write) is **never called** — because the tool was skipped. The write only fires when the tool actually ran. This is natural ADK framework behaviour and is what makes the pattern correct: you never overwrite a cache entry with itself.

---
**Expected output — cache miss (first call):**
```
[CACHE] read_search_cache | tool: Google_Search_agent
[CACHE] Key: cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
[CACHE MISS] No cached result found. Proceeding with tool execution.
[CACHE] write_search_cache | storing result
[CACHE WRITE] Result cached successfully.
```
**Expected output — cache hit (repeated call):**
```
[CACHE] read_search_cache | tool: Google_Search_agent
[CACHE] Key: cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
[CACHE HIT] Returning cached result. Tool execution skipped.
(write_search_cache never fires)
```


## ⚙️ 1. Setup: Install Libraries

Pinning the version ensures our code will always work as expected.

In [ ]:
!pip install google-adk==1.29.0 -q

## 🔑 2. Authentication: Configure Your API Key

In [ ]:
import os
from getpass import getpass

api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully!")

## 🤖 3. Model Configuration

Define the model names once here. To upgrade to a newer model in the future,
change these two constants — nothing else in the notebook needs to touch.

In [ ]:
# ── Model Configuration ───────────────────────────────────────────────────────
# Change these two constants to swap models across the entire notebook.
# To upgrade to a newer model in the future, update AGENT_MODEL and JUDGE_MODEL here.

AGENT_MODEL = "gemini-2.5-flash"  # used by all workflow agents
JUDGE_MODEL = "gemini-2.5-flash"   # used by the safety judge

## 🪝 4. [CARRIED OVER from 8.3, 8.4 & 8.5] Define the Observability Callbacks

These two callbacks are unchanged from Lecture 8.3 and carried forward through 8.4, 8.5, and 8.6.
They fire at the **agent** boundary (entry and exit).

The new caching callbacks in the next cells fire at the **tool** boundary on `cost_cutter_agent`.

All six callbacks now coexist and fire simultaneously during a live run:
- `guardrail_before_workflow` — `before_agent_callback` on `budget_optimizer_workflow` (agent boundary)
- `log_agent_entry` — `before_agent_callback` on `spending_proposer_agent` (agent boundary)
- `sanitize_cost_cutter_response` — `after_model_callback` on `cost_cutter_agent` (model boundary)
- `audit_and_validate_sum_costs` — `before_tool_callback` on `accountant_agent` (tool boundary)
- `read_search_cache` — `before_tool_callback` on `cost_cutter_agent` (tool boundary) ← **8.7 NEW**
- `write_search_cache` — `after_tool_callback` on `cost_cutter_agent` (tool boundary) ← **8.7 NEW**
- `log_agent_exit` — `after_agent_callback` on `plan_retriever_agent` (agent boundary)


In [ ]:
from datetime import datetime
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types

# A module-level variable so log_agent_exit can calculate elapsed time.
_workflow_start_time: datetime = None


def log_agent_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    before_agent_callback for spending_proposer_agent.

    Fires once, right before the first LLM call in the entire workflow.
    Records the start time and prints a structured ENTRY log line.

    Returns None — the agent proceeds normally. Nothing is skipped.
    """
    global _workflow_start_time
    _workflow_start_time = datetime.now()  # Capture start time for later

    # --- Read from CallbackContext ---
    agent_name    = callback_context.agent_name      # e.g. 'spending_proposer_agent'
    invocation_id = callback_context.invocation_id   # unique UUID for this run
    state_keys    = list(callback_context.state.to_dict().keys())  # what's in memory so far
    timestamp     = _workflow_start_time.strftime("%H:%M:%S")

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[ENTRY] {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        state_keys : {state_keys}")
    print("="*60)

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — proceed with the agent as normal.'
    return None


def log_agent_exit(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    after_agent_callback for plan_retriever_agent.

    Fires once, right after the last agent in the workflow completes.
    Calculates total elapsed time and prints a structured EXIT log line.

    Returns None — the agent's output is used unchanged. Nothing is replaced.
    """
    now        = datetime.now()
    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    state_keys = list(callback_context.state.to_dict().keys())
    timestamp  = now.strftime("%H:%M:%S")

    # Calculate duration only if log_agent_entry ran first
    if _workflow_start_time is not None:
        elapsed = (now - _workflow_start_time).seconds
        duration_str = f"{elapsed}s"
    else:
        duration_str = "n/a"

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[EXIT]  {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        duration   : {duration_str}")
    print(f"        state_keys : {state_keys}")
    print("="*60 + "\n")

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — use the agent's real output as-is.'
    return None

## 🛡️ 5. [CARRIED OVER from 8.4] Define the LLM-as-Judge Guardrail

This cell is **unchanged from Lecture 8.4**. The guardrail fires at the **agent** boundary on `budget_optimizer_workflow` — before any sub-agent starts.

### Why it is still here

The 8.7 caching callbacks operate at the **tool** boundary on `cost_cutter_agent` — a completely different level. All callbacks coexist without interference:

| Callback | Hook | Fires on | Boundary |
|---|---|---|---|
| `guardrail_before_workflow` | `before_agent_callback` | `budget_optimizer_workflow` | Agent |
| `log_agent_entry` | `before_agent_callback` | `spending_proposer_agent` | Agent |
| `sanitize_cost_cutter_response` | `after_model_callback` | `cost_cutter_agent` | Model |
| `audit_and_validate_sum_costs` | `before_tool_callback` | `accountant_agent` | Tool |
| `read_search_cache` | `before_tool_callback` | `cost_cutter_agent` | Tool |
| `write_search_cache` | `after_tool_callback` | `cost_cutter_agent` | Tool |
| `log_agent_exit` | `after_agent_callback` | `plan_retriever_agent` | Agent |

### Return value contract (reminder)
- Return `None` → topic is safe, workflow runs normally
- Return `Content` → entire workflow cancelled instantly — no sub-agent ever starts


In [ ]:
# ============================================================
# Lecture 8.4 — LLM-as-Judge Guardrail  [CARRIED OVER — UNCHANGED]
# Design Patterns: Guardrails & Policy Enforcement (P1)
#                  Conditional Skipping of Steps (P6)
# ============================================================

from google.genai import client as genai_client

# Initialise a direct Gemini client for the safety judge.
# This is a raw API call — completely separate from the ADK runner.
safety_client = genai_client.Client()

# ── Judge prompts ─────────────────────────────────────────────────────────
# Two-stage design:
#   Prompt 1 — binary verdict (YES/NO). Fast, cheap, used on every request.
#   Prompt 2 — human-readable explanation. Only called when verdict is YES,
#              so clean topics pay no extra cost.

SAFETY_VERDICT_PROMPT = """
You are a strict safety officer for an event planning company.
Evaluate the following event planning request.

Does it involve any of the following:
- Dangerous or high-risk activities
- Weapons, arms, or military equipment
- Illegal substances or narcotics
- Illegal activities of any kind
- Activities that cannot be commercially insured
- Adult-only or explicit content
- Anything that exposes the company to legal or reputational risk

Reply with EXACTLY one word — either YES or NO.
No explanation. No punctuation. Just the single word.

Event request: "{topic}"
"""

SAFETY_REASON_PROMPT = """
You are a polite but firm safety officer for an event planning company.
A client has requested help planning an event, but it has been flagged as
unsafe or inappropriate for our business.

Write a short, professional refusal message (2-3 sentences) addressed to
the client. Explain specifically why this type of event falls outside what
the company can assist with. Be clear but courteous. Do not offer
workarounds or alternatives.

Event request: "{topic}"
"""


def guardrail_before_workflow(
    callback_context: CallbackContext,
) -> Optional[types.Content]:
    """
    before_agent_callback on budget_optimizer_workflow (the SequentialAgent).

    Two-stage LLM-as-judge:
      Stage 1 — fast binary verdict (YES/NO) on every request.
      Stage 2 — rich refusal explanation, only when Stage 1 says YES.

    The explanation is written into state["refusal_reason"] so the runner
    can surface it as the final response instead of "No plan found."

    Placed on the SequentialAgent so it fires once before ANY sub-agent
    starts. Returning Content cancels the entire workflow instantly.
    """
    topic = callback_context.state.get("topic", "")

    print("\n" + "─" * 60)
    print(f"[SAFETY JUDGE] Evaluating topic: '{topic}'")

    # ── Stage 1: Binary verdict ───────────────────────────────────────────
    verdict_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_VERDICT_PROMPT.format(topic=topic),
    )
    verdict = verdict_response.text.strip().upper()
    print(f"[SAFETY JUDGE] Verdict         : {verdict}")

    if "YES" not in verdict:
        print(f"  └─ ✅ SAFE — starting workflow.")
        print("─" * 60)
        return None                        # topic is safe, proceed normally

    # ── Stage 2: Rich explanation (only reached when blocked) ─────────────
    print(f"[SAFETY JUDGE] Generating refusal explanation...")
    reason_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_REASON_PROMPT.format(topic=topic),
    )
    refusal_reason = reason_response.text.strip()
    print(f"[SAFETY JUDGE] Reason          : {refusal_reason}")
    print(f"  └─ 🚫 BLOCKED — high-risk topic. Workflow cancelled.")
    print("─" * 60)

    # Write the rich explanation into session state.
    # The runner reads state["refusal_reason"] when state["final_presentation"]
    # is absent — this is how the final response reaches the user.
    callback_context.state["refusal_reason"] = refusal_reason

    # Return Content to cancel the entire workflow.
    return types.Content(
        role="model",
        parts=[types.Part(text=refusal_reason)],
    )

## 🧹 6. [CARRIED OVER from 8.5] Define the Response Sanitization Callback

This cell is **unchanged from Lecture 8.5**. The `after_model_callback` on `cost_cutter_agent` strips markdown fences and coerces string costs to floats before the framework processes the LLM response.

It fires at the **model** boundary. The new 8.7 caching callbacks fire at the **tool** boundary — a later, narrower interception point that sees the final argument values the LLM produced for the tool call.

| Callback | What it fixes | When it fires |
|---|---|---|
| `sanitize_cost_cutter_response` | Fence-wrapped JSON, string costs in plan | After `cost_cutter_agent` LLM responds |
| `read_search_cache` | Returns cached result instead of calling tool | Before `google_search_tool` executes |
| `write_search_cache` | Stores fresh result for future hits | After `google_search_tool` executes |


In [ ]:
# ============================================================
# Lecture 8.5 — Response Sanitization: after_model_callback
# Design Pattern: Request / Response Modification (P5)
# ============================================================

import re
import json
from google.adk.models import LlmResponse
from google.genai import types as genai_types


def sanitize_cost_cutter_response(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> Optional[LlmResponse]:
    """
    after_model_callback for cost_cutter_agent.

    Intercepts the raw LLM response and fixes two consistent formatting flaws:
      1. Markdown code fences   -- ```json ... ```  wrapping the JSON
      2. String costs           -- "cost": "5000" instead of "cost": 5000

    Return value contract:
      - Function call response       -> return None immediately (don't touch tool calls)
      - Fences or string costs found -> return a NEW LlmResponse with cleaned content
      - Already clean               -> return None (original passes through unchanged)

    Why return a new object rather than mutating the original?
    Other callbacks in the chain may hold references to the original LlmResponse.
    Mutating it in-place would silently affect those other callbacks.
    Always build a new one.
    """
    agent_name = callback_context.agent_name

    # -- Guard: only process text responses ---------------------------------
    # LLM responses can contain function calls, not just text.
    # Check before assuming parts[0] is a text part.
    if (
        not llm_response.content
        or not llm_response.content.parts
        or llm_response.content.parts[0].function_call is not None
    ):
        return None   # Tool call — pass through untouched

    raw_text = llm_response.content.parts[0].text
    if not raw_text:
        return None

    cleaned = raw_text
    modified = False
    coerce_count = 0

    # -- Fix 1: Strip markdown code fences ----------------------------------
    fence_pattern = r"^\s*```(?:json)?\s*\n?(.*?)\n?\s*```\s*$"
    fence_match = re.search(fence_pattern, cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
        modified = True
        print(f"[SANITIZE] Stripped markdown fences from {agent_name} response")

    # -- Fix 2: Coerce string costs to float --------------------------------
    # Walk the parsed JSON and convert any value whose key contains 'cost'
    # from a string to a float.
    try:
        data = json.loads(cleaned)

        def coerce_costs(obj):
            nonlocal coerce_count
            if isinstance(obj, dict):
                for key, value in obj.items():
                    if "cost" in key.lower() and isinstance(value, str):
                        try:
                            obj[key] = float(value)
                            coerce_count += 1
                        except ValueError:
                            pass   # leave non-numeric strings alone
                    else:
                        coerce_costs(value)
            elif isinstance(obj, list):
                for item in obj:
                    coerce_costs(item)

        coerce_costs(data)

        if coerce_count > 0:
            cleaned = json.dumps(data)
            modified = True
            print(f"[SANITIZE] Coerced {coerce_count} string cost(s) to float in {agent_name} response")

    except (json.JSONDecodeError, TypeError):
        # Not valid JSON — skip coercion, but still return cleaned text
        # if fences were stripped
        pass

    # -- Return -------------------------------------------------------------
    if not modified:
        print(f"[SANITIZE] Response already clean. Passing through.")
        return None   # No change needed — return None so original is used

    # Build a NEW LlmResponse — never mutate the original
    cleaned_content = genai_types.Content(
        role=llm_response.content.role,
        parts=[genai_types.Part(text=cleaned)],
    )
    return LlmResponse(content=cleaned_content)

## 🔍 7. [CARRIED OVER from 8.6] High-Cost Threshold & Tool Auditing Callback

These two cells are **unchanged from Lecture 8.6**. The `HIGH_COST_THRESHOLD` constant and `audit_and_validate_sum_costs` are untouched.


In [ ]:
# ── 8.6 Constant (unchanged) ──────────────────────────────────────────────────
# Costs above this value trigger a [FLAG] warning log line in the auditor.
# The callback still proceeds — this is observation-only, not blocking.

HIGH_COST_THRESHOLD = 10_000   # configurable — set low enough to flag luxury venue/catering costs in the workflow

In [ ]:
# ============================================================
# Lecture 8.6 — Tool Auditing and Argument Validation  [CARRIED OVER — UNCHANGED]
# Design Pattern: Request / Response Modification (P5) at the tool layer
# ============================================================

from typing import Any, Dict
from google.adk.tools import ToolContext
from google.adk.tools.base_tool import BaseTool

# Module-level iteration counter — tracks which loop pass triggered the tool call.
# Reset to 0 each time you run a fresh workflow.
_sum_costs_iteration: int = 0


def audit_and_validate_sum_costs(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
) -> Optional[Dict]:
    """
    before_tool_callback for accountant_agent, applied to sum_costs.

    Delivers three capabilities in a single pass:
      1. Audit trail   — logs every invocation with costs list and iteration number
      2. Validation    — silently removes negative or non-numeric values before tool runs
      3. High-cost flag — warns when any single cost exceeds HIGH_COST_THRESHOLD

    Return value contract:
      Always returns None — this callback never blocks execution.
      When invalid values are found, args['costs'] is mutated in-place.
      The tool then receives the cleaned argument dict automatically.
    """
    global _sum_costs_iteration

    # -- Defensive guard: only act on sum_costs ---------------------------
    if tool.name != "sum_costs":
        return None

    _sum_costs_iteration += 1
    agent_name = tool_context.agent_name
    costs = args.get("costs", [])

    # -- 1. Audit trail ---------------------------------------------------
    print(f"[AUDIT] sum_costs called | agent: {agent_name} | iteration: {_sum_costs_iteration}")
    print(f"[AUDIT] costs submitted: {costs}")

    # -- 2. Argument validation and sanitization --------------------------
    cleaned = []
    for value in costs:
        # Check: must be numeric
        if not isinstance(value, (int, float)):
            print(f"[VALIDATE] Removed invalid value: {value!r} (non-numeric)")
            continue
        # Check: must be non-negative
        if value < 0:
            print(f"[VALIDATE] Removed invalid value: {value} (negative)")
            continue
        cleaned.append(value)

    if len(cleaned) != len(costs):
        args["costs"] = cleaned   # mutate in-place — tool receives cleaned list
        print(f"[AUDIT] Sanitized costs: {cleaned}")

    # -- 3. High-cost flagging --------------------------------------------
    flagged = False
    for value in args.get("costs", []):
        if isinstance(value, (int, float)) and value > HIGH_COST_THRESHOLD:
            print(f"[FLAG] Suspiciously high cost detected: {value} (threshold: {HIGH_COST_THRESHOLD})")
            flagged = True

    if flagged:
        print(f"[AUDIT] Proceeding with flagged costs.")
    elif len(cleaned) == len(costs):
        # Only print this if no sanitization happened and no flags were raised
        print(f"[AUDIT] All values valid. Proceeding.")

    # Always return None — never block tool execution
    return None

## 🗝️ 8. [NEW] Cache Key Helper

A single shared helper function generates the cache key used by **both** the read and write callbacks.

### Why a shared helper?

The read and write callbacks are two halves of one mechanism. If they generate keys differently, a miss during read can never be found on write — and vice versa. Centralising key construction in one function guarantees they always agree.

### Key anatomy

```
cache:Google_Search_agent:{"query": "affordable catering New York 50 people"}
│     │                   │
│     │                   └─ serialised args — differentiates between queries
│     └─ tool name — differentiates between tools (future-proofing)
└─ namespace prefix — separates all cache entries from other state keys
```


In [ ]:
# ============================================================
# Lecture 8.7 — Cache Key Helper  [NEW]
# ============================================================

import json


def _make_cache_key(tool_name: str, args: Dict[str, Any]) -> str:
    """
    Build a deterministic, namespaced cache key from a tool name and its arguments.

    Format:  cache:<tool_name>:<json_serialised_args>

    Design decisions:
      - 'cache:' prefix namespaces all cache entries away from other state keys
        (e.g. 'budget', 'topic', 'current_plan').
      - tool_name is included so a future cache serving multiple tools never
        collides across tool types.
      - The full args dict serialised as JSON means two different queries to
        the same tool always produce different keys.

    Example outputs:
      cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
      cache:Google_Search_agent:{"query": "budget catering New York under 3000"}
    """



# Quick sanity check — run this cell to confirm key generation is deterministic
key1 = _make_cache_key("Google_Search_agent", {"query": "affordable venue New York"})
key2 = _make_cache_key("Google_Search_agent", {"query": "affordable venue New York"})
print(f"Key 1 : {key1}")
print(f"Key 2 : {key2}")
print(f"Match : {key1 == key2}  ← must be True")

## 📖 9. [NEW] Define the Cache Read Callback (`before_tool_callback`)

`read_search_cache` is the **before_tool_callback** — it fires after the LLM has decided to call `google_search_tool` and has produced the query argument, but before the tool executes.

### Callback signature

```python
def read_search_cache(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext
) -> Optional[Dict]:
```

### Return value contract

| Condition | Return value | What happens |
|---|---|---|
| Wrong tool name | `None` immediately | Defensive guard — fires for every tool on the agent |
| Cache miss | `None` | Tool executes normally; `write_search_cache` fires after |
| Cache hit | The cached `dict` | Tool is **skipped entirely**; `write_search_cache` never fires |

### The key insight

Returning a non-`None` dict from a `before_tool_callback` **replaces the tool's return value** — the tool function never runs. This is the same return-value mechanism used in Lecture 8.4 to skip agent execution entirely. Here, instead of blocking for policy reasons, we are blocking for performance reasons: we already have the answer.


In [ ]:
# ============================================================
# Lecture 8.7 — Cache Read: before_tool_callback  [NEW]
# Design Pattern: Caching (P4)
# ============================================================


def read_search_cache(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
) -> Optional[Dict]:
    """
    before_tool_callback for cost_cutter_agent, applied to google_search_tool.

    Implements the READ half of a read-through cache backed by session state.

    Execution flow:
      1. Generate a deterministic cache key from tool name + serialised args.
      2. Check tool_context.state for an existing entry under that key.
      3. Cache HIT  → log [CACHE HIT], return the cached dict.
                       The ADK framework uses the returned dict as the tool's
                       result — the tool function never executes.
                       write_search_cache never fires (tool was skipped).
      4. Cache MISS → log [CACHE MISS], return None.
                       The tool executes normally.
                       write_search_cache fires afterwards to store the result.

    Return value contract:
      - None         → cache miss; tool runs; write callback fires after
      - Dict         → cache hit;  tool skipped; write callback never fires

    Why tool_context.state as the cache store?
      Session state persists across all agents within a session and across
      all iterations of the LoopAgent. It is the natural shared memory in ADK.
      Using it as a cache store requires no external infrastructure — just
      a namespaced key convention to avoid collisions with other state entries.
    """
    # -- Defensive guard: only cache google_search_tool calls ---------------


    print(f"[CACHE] read_search_cache | tool: {tool.name}")
    print(f"[CACHE] Key: {cache_key}")

    # -- Cache lookup -------------------------------------------------------

    # None return allows tool to execute normally


## 📝 10. [NEW] Define the Cache Write Callback (`after_tool_callback`)

`write_search_cache` is the **after_tool_callback** — it fires after `google_search_tool` has executed and produced a result. It stores that result in session state so future calls with the same query return instantly.

### Callback signature

`after_tool_callback` takes **four arguments** — including a new one we have not seen before:

```python
def write_search_cache(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
    tool_response: Dict          # ← NEW: the result the tool just produced
) -> Optional[Dict]:
```

### Return value contract

| Return value | What happens |
|---|---|
| `None` | Original `tool_response` passes through unchanged to the LLM |
| A new `Dict` | Replaces `tool_response` — the LLM sees the returned dict instead |

We always return `None` here. The write callback only needs to store the result — it never needs to modify it.

### The asymmetry — why this callback only fires on misses

When `read_search_cache` returns a cached dict (cache hit), the ADK framework treats that as the complete tool result and **does not call the tool function at all**. Because `after_tool_callback` fires only when the tool function actually ran, it never fires on hits. This is not special logic we had to write — it is the natural framework behaviour that makes the pattern correct.


In [ ]:
# ============================================================
# Lecture 8.7 — Cache Write: after_tool_callback  [NEW]
# Design Pattern: Caching (P4)
# ============================================================


def write_search_cache(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
    tool_response: Dict,
) -> Optional[Dict]:
    """
    after_tool_callback for cost_cutter_agent, applied to google_search_tool.

    Implements the WRITE half of a read-through cache backed by session state.

    Execution flow:
      1. Defensive guard — only cache google_search_tool results.
      2. Regenerate the same cache key using the same helper function.
      3. Store tool_response in state under that key.
      4. Return None — the original tool_response passes through unchanged.

    This callback only fires when the tool actually ran (i.e. on a cache miss).
    On a cache hit, read_search_cache returned the cached result directly and
    the tool was skipped — so after_tool_callback never fires.
    This asymmetry is natural ADK framework behaviour, not special logic here.

    Return value contract:
      Always returns None — the original tool_response is used unchanged.
      We store it, we don't modify it.

    Why the same _make_cache_key call?
      The key must be identical to the one read_search_cache generated for the
      same (tool, args) pair. Using the shared helper guarantees this.
    """
    # -- Defensive guard: only cache google_search_tool results -------------


    print(f"[CACHE] write_search_cache | storing result under key:")
    print(f"[CACHE] Key: {cache_key}")

    # -- Store result in session state -------------------------------------


    # Return None — original tool_response passes through to the LLM unchanged
    return None

## 🛠️ 11. Define Workflow Tools

**Unchanged from Lecture 8.6.** The caching callbacks are attached to `cost_cutter_agent`, not to the tool definitions.


In [ ]:
from google.adk.tools import ToolContext

def sum_costs(costs: list[float]) -> float:
    """Calculates the sum of a list of numbers."""
    print(f"  [Tool Call] sum_costs on the list: {costs}")
    return sum(costs)

def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the plan is approved and within budget."""
    print(f"  [Tool Call] Budget approved. Terminating loop: {json.dumps(tool_context.state.to_dict())}")
    tool_context.actions.escalate = True
    return None

## 12. Create Tool Wrappers

Unchanged from Lecture 8.6.


In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_agent",
    model=AGENT_MODEL,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)

google_search_tool = AgentTool(agent=google_search_agent)

## 📝 13. Create Agents

**One change from Lecture 8.6** — highlighted with `# <- 8.7 CHANGE`:

- `cost_cutter_agent` — gains `before_tool_callback=read_search_cache` and `after_tool_callback=write_search_cache`
- `spending_proposer_agent`, `accountant_agent`, `plan_retriever_agent` — all unchanged from 8.6

Six callbacks now fire across the workflow simultaneously:
- `guardrail_before_workflow` — on `budget_optimizer_workflow` (before_agent_callback)
- `log_agent_entry` — on `spending_proposer_agent` (before_agent_callback)
- `sanitize_cost_cutter_response` — on `cost_cutter_agent` (after_model_callback)
- `audit_and_validate_sum_costs` — on `accountant_agent` (before_tool_callback)
- `read_search_cache` — on `cost_cutter_agent` (before_tool_callback) ← **8.7 CHANGE**
- `write_search_cache` — on `cost_cutter_agent` (after_tool_callback) ← **8.7 CHANGE**
- `log_agent_exit` — on `plan_retriever_agent` (after_agent_callback)


In [ ]:
COMPLETION_PHRASE = "The plan is within the budget."

# Agent 1: Proposes the initial, expensive plan (runs once).
# before_agent_callback=log_agent_entry carried over from 8.3 unchanged.
spending_proposer_agent = Agent(
    name="spending_proposer_agent",
    model=AGENT_MODEL,
    tools=[google_search],
    instruction="""
    You are a luxury event planner. For a {{topic}}, find a high-end venue and a gourmet catering service.

    Output a JSON object with items and their estimated costs, like:
    {"venue": {"name": "The Ritz London", "cost": 10000}, "catering": {"name": "Gourmet Chefs Inc.", "cost": 5000}}
    """,
    output_key="current_plan",
    before_agent_callback=log_agent_entry,   # <- 8.3/8.4/8.5/8.6 (unchanged)
)

# Agent 2 (in loop): The "Accountant" that critiques the plan.
# before_tool_callback added in 8.6 — unchanged.
accountant_agent = Agent(
    name="accountant_agent",
    model=AGENT_MODEL,
    tools=[sum_costs],
    before_tool_callback=audit_and_validate_sum_costs,   # <- 8.6 (unchanged)
    instruction=f"""
    You are a meticulous accountant. Your budget is {{{{budget}}}}.
    The current plan is: {{{{current_plan}}}}

    Extract the costs from the plan and use the `sum_costs` tool to get the total.
    - IF the total cost is > {{{{budget}}}}, output a critique like: "This plan is over budget by [amount]. Find a cheaper [item]."
    - ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key="critique",
)

# Agent 3 (in loop): The "Cost Cutter" that refines the plan.
# after_model_callback from 8.5 — unchanged.
# before_tool_callback + after_tool_callback added in 8.7 for caching.
cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=AGENT_MODEL,
    tools=[google_search_tool, exit_loop],
    instruction=f"""
    You are a cost-cutting expert. You must refine a plan based on a critique.
    The critique is: {{{{critique}}}}
    The current plan is: {{{{current_plan}}}}

    - IF the critique is '{COMPLETION_PHRASE}'
        1. You MUST call the `exit_loop` tool with no arguments.
        2. After calling exit_loop, output the current plan EXACTLY as-is, character for character,
           with no modifications, no acknowledgements, no commentary, and no extra text. Do not summarize it.
           Do not rephrase it. Do not add "Budget approved" or any other text.
           Just echo {{{{current_plan}}}} verbatim.
    - ELSE, read the critique to identify the overpriced item. Use your search tool to find a cheaper alternative for that item.
      Output a new JSON object with the updated plan.
    """,
    output_key="current_plan",
    after_model_callback=sanitize_cost_cutter_response,   # <- 8.5 (unchanged)
    before_tool_callback=read_search_cache,               # <- 8.7 CHANGE
    after_tool_callback=write_search_cache,               # <- 8.7 CHANGE
)

# Agent 4: Presents the final approved plan (runs once after loop).
# Unchanged from 8.4.
plan_retriever_agent = Agent(
    name="plan_retriever_agent",
    model=AGENT_MODEL,
    instruction="""
    You are a plan finalizer. Your only job is to present the final, approved plan.
    The plan is available in the context variable `{{current_plan}}`.

    Your output must be the content of the final plan presented in a clear and easy-to-read format.
    """,
    tools=[],
    output_key="final_presentation",
    after_agent_callback=log_agent_exit,   # <- 8.3/8.4/8.5/8.6 (unchanged)
)

## 🔄 14. Assemble the Loop and Sequential Workflows

Unchanged from Lecture 8.6. The caching callbacks live on `cost_cutter_agent` inside the loop — no changes needed to the LoopAgent or SequentialAgent definitions.


In [ ]:
from google.adk.agents import SequentialAgent, LoopAgent

# Unchanged from Section 5.
budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3,
)

# <- 8.4 (unchanged): guardrail still lives on the SequentialAgent.
# 8.7 changes are on cost_cutter_agent, not here.
budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[spending_proposer_agent, budget_refinement_loop, plan_retriever_agent],
    before_agent_callback=guardrail_before_workflow,   # <- 8.4 (unchanged)
)

## 🚀 15. Build the Execution Engine

Unchanged from Lecture 8.6. The caching callbacks fire automatically inside the loop — the runner does not need to know about them.


In [ ]:
from IPython.display import display, Markdown

from google.adk.sessions import Session
from google.genai.types import Content, Part
from google.adk.runners import Runner

async def run_agent_query(agent: Agent, query: str, topic: str, budget: str, session: Session, user_id: str):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
            state_delta={"budget": budget, "topic": topic, "COMPLETION_PHRASE": COMPLETION_PHRASE}
        ):
            pass
    except Exception as e:
        final_response = f"An error occurred: {e}"
        return final_response

    # Read the final response from session state.
    #
    # Two possible paths through the workflow:
    #
    #   ✅ CLEAN topic  → plan_retriever_agent runs and writes final_presentation.
    #                     We read that.
    #
    #   🚫 BLOCKED topic → guardrail cancels the workflow and writes refusal_reason
    #                      into state. plan_retriever never runs, so
    #                      final_presentation is never written. We read
    #                      refusal_reason instead.
    #
    final_session = await session_service.get_session(
        app_name=agent.name,
        user_id=user_id,
        session_id=session.id
    )
    state = final_session.state

    if "final_presentation" in state:
        final_response = state["final_presentation"]
    elif "refusal_reason" in state:
        final_response = state["refusal_reason"]
    else:
        final_response = "No response was generated."

    print("\n" + "-"*50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-"*50 + "\n")

    return final_response

## ✨ 16. Initialize Session Service

Unchanged from Lecture 8.6.


In [ ]:
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
user_id = "adk_event_planner_001"

## 🔬 17. Standalone Cache Demo

Before running the full workflow, let's verify both callbacks behave correctly by calling them directly with mock inputs.

This cell demonstrates all **four cache scenarios** — no runner, no session, no agents required.

**Scenario 1** — First call (cache miss, write fires)
**Scenario 2** — Repeated call, same query (cache hit, tool skipped, write never fires)
**Scenario 3** — Different query (cache miss, different key, write fires)
**Scenario 4** — Full loop simulation showing cache hits accumulating across iterations


In [ ]:
# ============================================================
# Standalone demo — verify cache read/write with mock objects
# ============================================================

class _MockState(dict):
    """A real dict that also supports .get() — used as ctx.state."""
    pass

class _MockTool:
    """Minimal stand-in for BaseTool — only the name attribute is needed."""
    def __init__(self, name):
        self.name = name

class _MockToolContext:
    """
    Minimal stand-in for ToolContext.

    Uses a real dict for state so that:
        tool_context.state.get(key)      — works correctly (returns None on miss)
        tool_context.state[key] = value  — works correctly (stores the value)

    MagicMock cannot be used here: Python looks up dunder methods (__setitem__)
    on the *class*, not the instance, so assigning a lambda to the instance
    attribute is silently ignored and self gets injected as an extra argument.
    A plain dict subclass avoids the problem entirely.
    """
    def __init__(self, agent_name="cost_cutter_agent"):
        self.agent_name = agent_name
        self.state = _MockState()

def make_mock_tool(name):
    """Build a minimal mock BaseTool with just a name attribute."""
    return _MockTool(name)

def make_mock_tool_context():
    """Build a minimal mock ToolContext backed by a real dict."""
    return _MockToolContext()

mock_tool = make_mock_tool("Google_Search_agent")
mock_ctx  = make_mock_tool_context()

FAKE_RESULT   = {"results": [{"title": "Affordable Venue NYC",  "url": "https://example.com"}]}
FAKE_RESULT_2 = {"results": [{"title": "Budget Caterer NYC",    "url": "https://caterer.com"}]}

# ── Scenario 1 — First call: cache miss, write fires ──────────────────────────
print("=" * 65)
print("SCENARIO 1 — First call (cache miss → tool runs → write fires)")
print("=" * 65)
args1 = {"query": "affordable venue New York 50 people"}

result = read_search_cache(mock_tool, args1, mock_ctx)
print(f"→ read_search_cache returned: {result}  (None = proceed with tool)")

print("  [Tool Call] google_search_tool executes... (simulated ~2-3 seconds)")

write_search_cache(mock_tool, args1, mock_ctx, FAKE_RESULT)
print(f"→ write_search_cache called: result stored in state")

# ── Scenario 2 — Same query: cache hit, tool skipped, write never fires ───────
print()
print("=" * 65)
print("SCENARIO 2 — Repeated call, same query (cache hit → tool skipped)")
print("=" * 65)
args2 = {"query": "affordable venue New York 50 people"}   # identical to args1

result = read_search_cache(mock_tool, args2, mock_ctx)
print(f"→ read_search_cache returned: {result}")
print(f"  (Non-None return = tool skipped entirely; write_search_cache never fires)")

# ── Scenario 3 — Different query: cache miss with a new key ───────────────────
print()
print("=" * 65)
print("SCENARIO 3 — Different query (cache miss → new key → write fires)")
print("=" * 65)
args3 = {"query": "budget catering New York under 3000"}   # different from args1

result = read_search_cache(mock_tool, args3, mock_ctx)
print(f"→ read_search_cache returned: {result}  (None = proceed with tool)")

print("  [Tool Call] google_search_tool executes... (simulated)")
write_search_cache(mock_tool, args3, mock_ctx, FAKE_RESULT_2)
print(f"→ write_search_cache called: second key stored")

# ── Scenario 4 — Loop simulation: misses in iter 1, hits in iters 2 & 3 ──────
print()
print("=" * 65)
print("SCENARIO 4 — Loop simulation: hits accumulate across iterations")
print("=" * 65)

loop_ctx      = make_mock_tool_context()
venue_query   = {"query": "cheaper venue New York"}
catering_query = {"query": "affordable catering NYC"}
dj_query      = {"query": "budget DJ New York"}

for iteration in range(1, 4):
    print(f"\n=== Loop Iteration {iteration} ===")

    # Venue query — miss in iteration 1, hit in 2 & 3
    r = read_search_cache(mock_tool, venue_query, loop_ctx)
    if r is None:
        print("  → venue: tool runs")
        write_search_cache(mock_tool, venue_query, loop_ctx, {"results": ["Venue result"]})
    else:
        print("  → venue: instant cache return (no API call)")

    # Catering query — asked once in iteration 1 only
    if iteration == 1:
        r = read_search_cache(mock_tool, catering_query, loop_ctx)
        if r is None:
            print("  → catering: tool runs")
            write_search_cache(mock_tool, catering_query, loop_ctx, {"results": ["Catering result"]})

    # DJ query — miss in iteration 2, hit in iteration 3
    if iteration >= 2:
        r = read_search_cache(mock_tool, dj_query, loop_ctx)
        if r is None:
            print("  → DJ: tool runs")
            write_search_cache(mock_tool, dj_query, loop_ctx, {"results": ["DJ result"]})
        else:
            print("  → DJ: instant cache return (no API call)")


## ▶️ 18a. Run — CLEAN Topic (Full Workflow with Caching)

The topic `"50 person AI event in New York"` passes the guardrail and runs the full workflow.

Watch for:
- **`[CACHE MISS]`** lines on every `google_search_tool` call — the cache is cold, all searches execute and are stored in session state
- **`[CACHE WRITE]`** confirming each result is stored under its key
- The cache infrastructure is fully active: every search goes through `read_search_cache` → tool → `write_search_cache`

**A note on cache hits in this workflow:**
In this refinement loop, `cost_cutter_agent` deliberately searches for *different* things on each iteration — a cheaper venue, then cheaper catering, then a cheaper DJ. Because the LLM generates a different query string each time, you will see `[CACHE MISS]` on every call here.

That is expected and correct behaviour. The cache pays off in two real scenarios:
1. The LLM happens to repeat an identical query across iterations (which does occur in practice)
2. The same workflow is run again in the same session — all previously seen queries return instantly

The standalone demo in Cell 17 already proved the hit path works correctly. Here we are confirming the full callback stack fires cleanly on a real workflow run.

You will also see all prior callback layers active simultaneously:
- `[SAFETY JUDGE]` — guardrail evaluates the topic
- `[ENTRY]` — `log_agent_entry` fires as `spending_proposer_agent` starts
- `[SANITIZE]` — `sanitize_cost_cutter_response` fires after each `cost_cutter_agent` LLM call
- `[AUDIT]` — `audit_and_validate_sum_costs` fires before each `sum_costs` tool call
- `[CACHE]` — `read_search_cache` + `write_search_cache` fire on each `google_search_tool` call ← **NEW**
- `[EXIT]` — `log_agent_exit` fires after `plan_retriever_agent` completes


In [ ]:
import time

# Reset the auditing iteration counter before the live run
_sum_costs_iteration = 0

async def run_clean_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "50 person AI event in New York"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")

    t_start = time.time()
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)
    t_end = time.time()

    print(f"\n⏱️  Total workflow time: {t_end - t_start:.1f}s")
    print("   (Observe: iteration 1 is slowest — cache is cold.")
    print("    Iterations 2+ return repeated queries instantly — cache is warm.)")

await run_clean_topic()

## 🚫 18b. Run — BLOCKED Topic (Guardrail Still Intercepts)

The topic `"weapons convention"` is still blocked by the 8.4 guardrail.

Notice: `[CACHE]` lines never appear — because `cost_cutter_agent` never runs when the workflow is cancelled at the outermost boundary. This confirms the caching callbacks only fire when `google_search_tool` is actually called.


In [ ]:
async def run_blocked_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "weapons convention"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_blocked_topic()